In [2]:
# ============================================================
# CELL 1: IMPORT VÀ ĐỌC DỮ LIỆU
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

# Đường dẫn (tính từ data_processing/)
INPUT_PATH  = "../data/processed/ai_ready.csv"
OUTPUT_DIR  = "../data/splits/"

# Đọc dữ liệu
df = pd.read_csv(INPUT_PATH)

print("=" * 60)
print("THÔNG TIN DỮ LIỆU AI_READY")
print("=" * 60)
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"\nDanh sách cột:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# Kiểm tra kiểu dữ liệu
print(f"\nKiểu dữ liệu:")
print(df.dtypes.to_string())

# Kiểm tra NaN
print(f"\nTỷ lệ NaN:")
nan_pct = (df.isna().sum() / len(df) * 100).round(2)
for col in df.columns:
    if nan_pct[col] > 0:
        print(f"  {col:22s}: {nan_pct[col]:>5.2f}%")

# Phân bố nhãn
print(f"\nPhân bố nhãn mortality:")
vc = df["mortality"].value_counts()
vc_pct = df["mortality"].value_counts(normalize=True) * 100
for k in vc.index:
    label = "Alive" if k == 0 else "Expired"
    print(f"  {k} ({label}): {vc[k]:,} ({vc_pct[k]:.2f}%)")

# 5 dòng đầu
print(f"\n5 dòng đầu tiên:")
print(df.head())
display(df.head())

THÔNG TIN DỮ LIỆU AI_READY
Kích thước: 2,326 dòng × 25 cột

Danh sách cột:
   1. patientunitstayid
   2. age
   3. male_sex
   4. bmi
   5. mortality
   6. gcs
   7. heartrate_mean
   8. heartrate_min
   9. heartrate_max
  10. systemicmean_max
  11. sao2_mean
  12. sao2_min
  13. respiration_mean
  14. respiration_max
  15. temperature_mean
  16. creatinine_max
  17. bun_mean
  18. lactate_mean
  19. lactate_max
  20. wbc_mean
  21. hemoglobin_mean
  22. platelets_mean
  23. bilirubin_mean
  24. glucose_mean
  25. glucose_max

Kiểu dữ liệu:
patientunitstayid      int64
age                  float64
male_sex             float64
bmi                  float64
mortality              int64
gcs                  float64
heartrate_mean       float64
heartrate_min        float64
heartrate_max        float64
systemicmean_max     float64
sao2_mean            float64
sao2_min             float64
respiration_mean     float64
respiration_max      float64
temperature_mean     float64
creatinine_max    

,patientunitstayid,age,male_sex,bmi,mortality,gcs,heartrate_mean,heartrate_min,heartrate_max,systemicmean_max,...,creatinine_max,bun_mean,lactate_mean,lactate_max,wbc_mean,hemoglobin_mean,platelets_mean,bilirubin_mean,glucose_mean,glucose_max
0,141764,87.0,0.0,NaN,0,NaN,106.652174,90.0,138.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,141765,87.0,0.0,18.745276,0,15.0,83.190972,72.0,116.0,NaN,...,1.04,28.0,NaN,NaN,10.2,12.3,191.0,NaN,75.0,89.0
2,143870,76.0,1.0,27.788734,0,15.0,45.449367,40.0,55.0,79.0,...,1.14,13.5,NaN,NaN,11.7,12.3,152.0,NaN,131.5,140.0
3,144815,34.0,0.0,20.217741,0,15.0,83.800000,68.0,118.0,NaN,...,0.63,6.0,NaN,NaN,7.9,12.3,319.0,0.5,82.0,82.0
4,145427,61.0,1.0,29.007201,0,15.0,65.457983,49.0,85.0,NaN,...,1.05,14.0,NaN,NaN,21.1,13.3,196.0,NaN,128.5,139.0


In [3]:
# ============================================================
# CELL 2: TÁCH X VÀ y
# ============================================================

# y = nhãn cần dự đoán
y = df["mortality"].copy()

# X = 20 đặc trưng (bỏ patientunitstayid và mortality)
FEATURE_COLS = [
    # Nhân khẩu học (3)
    "age", "male_sex", "bmi",
    # Thần kinh (1)
    "gcs",
    # Sinh hiệu 24h đầu (9)
    "heartrate_mean", "heartrate_min", "heartrate_max",
    "systemicmean_max",
    "sao2_mean", "sao2_min",
    "respiration_mean", "respiration_max",
    "temperature_mean",
    # Xét nghiệm 24h đầu (10)
    "creatinine_max", "bun_mean",
    "lactate_mean", "lactate_max",
    "wbc_mean", "hemoglobin_mean", "platelets_mean",
    "bilirubin_mean",
    "glucose_mean", "glucose_max",
]

X = df[FEATURE_COLS].copy()

print("=" * 60)
print("TÁCH X VÀ y")
print("=" * 60)
print(f"X shape: {X.shape}  (kỳ vọng: 2326 × 20)")
print(f"y shape: {y.shape}  (kỳ vọng: 2326,)")
print(f"\nSố đặc trưng: {len(FEATURE_COLS)}")
print(f"Số đặc trưng khớp: {X.shape[1] == len(FEATURE_COLS)}")

# Kiểm tra y không có NaN
assert y.isna().sum() == 0, "y có NaN!"
print(f"✅ y không có NaN")

# Kiểm tra không còn cột ID trong X
assert "patientunitstayid" not in X.columns, "X còn chứa ID!"
print(f"✅ X không chứa patientunitstayid")

# Phân bố nhãn
print(f"\nPhân bố y:")
print(y.value_counts().to_string())

TÁCH X VÀ y
X shape: (2326, 23)  (kỳ vọng: 2326 × 20)
y shape: (2326,)  (kỳ vọng: 2326,)

Số đặc trưng: 23
Số đặc trưng khớp: True
✅ y không có NaN
✅ X không chứa patientunitstayid

Phân bố y:
mortality
0    2135
1     191


In [4]:
# ============================================================
# CELL 3: SPLIT TRAIN/TEST (80/20, STRATIFY)
# ============================================================

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,        # 20% test
    stratify=y,            # giữ tỷ lệ 8.21% tử vong ở cả 2 tập
    random_state=42        # tái lập
)

print("=" * 60)
print("SPLIT TRAIN/TEST")
print("=" * 60)
print(f"X_train: {X_train.shape}  (kỳ vọng: ~1861 × 23)")
print(f"X_test : {X_test.shape}   (kỳ vọng: ~465 × 23)")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

# Kiểm tra tỷ lệ nhãn ở 2 tập
print(f"\nTỷ lệ nhãn ở TRAIN:")
vc_train = y_train.value_counts(normalize=True) * 100
print(f"  0 (Alive): {vc_train.get(0, 0):.2f}%")
print(f"  1 (Expired): {vc_train.get(1, 0):.2f}%")

print(f"\nTỷ lệ nhãn ở TEST:")
vc_test = y_test.value_counts(normalize=True) * 100
print(f"  0 (Alive): {vc_test.get(0, 0):.2f}%")
print(f"  1 (Expired): {vc_test.get(1, 0):.2f}%")

# Số ca tử vong thực tế
print(f"\nSố ca tử vong:")
print(f"  Train: {y_train.sum()} / {len(y_train)}")
print(f"  Test : {y_test.sum()} / {len(y_test)}")

SPLIT TRAIN/TEST
X_train: (1860, 23)  (kỳ vọng: ~1861 × 23)
X_test : (466, 23)   (kỳ vọng: ~465 × 23)
y_train: (1860,)
y_test : (466,)

Tỷ lệ nhãn ở TRAIN:
  0 (Alive): 91.77%
  1 (Expired): 8.23%

Tỷ lệ nhãn ở TEST:
  0 (Alive): 91.85%
  1 (Expired): 8.15%

Số ca tử vong:
  Train: 153 / 1860
  Test : 38 / 466


In [5]:
# ============================================================
# CELL 4: TÍNH SCALE_POS_WEIGHT VÀ LƯU 4 FILE
# ============================================================

# ---------- Tính scale_pos_weight từ y_train ----------
n_alive_train   = (y_train == 0).sum()
n_expired_train = (y_train == 1).sum()
scale_pos_weight = n_alive_train / n_expired_train

print("=" * 60)
print("SCALE_POS_WEIGHT")
print("=" * 60)
print(f"Số ca Alive   (train): {n_alive_train:,}")
print(f"Số ca Expired (train): {n_expired_train:,}")
print(f"scale_pos_weight     : {scale_pos_weight:.4f}")
print(f"→ Dùng trong XGBoost: scale_pos_weight={scale_pos_weight:.4f}")

# ---------- Lưu 4 file ----------
OUTPUT_DIR = "../data/splits/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(f"{OUTPUT_DIR}X_train.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}y_train.csv", index=False)
X_test.to_csv(f"{OUTPUT_DIR}X_test.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}y_test.csv", index=False)

print(f"\n✅ Đã lưu 4 file vào: {OUTPUT_DIR}")
print(f"  X_train.csv: {X_train.shape}")
print(f"  y_train.csv: {y_train.shape}")
print(f"  X_test.csv : {X_test.shape}")
print(f"  y_test.csv : {y_test.shape}")

# ---------- Kiểm tra đọc lại ----------
print(f"\nKiểm tra đọc lại:")
X_train_check = pd.read_csv(f"{OUTPUT_DIR}X_train.csv")
y_train_check = pd.read_csv(f"{OUTPUT_DIR}y_train.csv")
print(f"  X_train đọc lại: {X_train_check.shape}")
print(f"  y_train đọc lại: {y_train_check.shape}")
print(f"  Cột y_train: {list(y_train_check.columns)}")

SCALE_POS_WEIGHT
Số ca Alive   (train): 1,707
Số ca Expired (train): 153
scale_pos_weight     : 11.1569
→ Dùng trong XGBoost: scale_pos_weight=11.1569

✅ Đã lưu 4 file vào: ../data/splits/
  X_train.csv: (1860, 23)
  y_train.csv: (1860,)
  X_test.csv : (466, 23)
  y_test.csv : (466,)

Kiểm tra đọc lại:
  X_train đọc lại: (1860, 23)
  y_train đọc lại: (1860, 1)
  Cột y_train: ['mortality']
